# Analise de Dados

Notebook com analises dos dados coletados.

**Analises:**
1. Estatisticas Descritivas
2. Analise de Volatilidade
3. Correlacoes
4. Retornos e Variacoes
5. Expectativas de Mercado
6. Mercado de Trabalho (IPEA + CAGED)

## 1. Setup

In [ ]:
# Explorers
from core.data import sgs, caged, expectations, ipea
from core.data import QueryEngine

# Analise
import pandas as pd
import numpy as np
from scipy import stats

# Visualizacao
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 10
sns.set_palette('husl')

qe = QueryEngine()

## 2. Estatisticas Descritivas

In [ ]:
# Carregar dados SGS - Daily
df_daily = sgs.read('selic', 'cdi', 'dolar_ptax', 'euro_ptax', start='2000')

print("SGS DAILY - Estatisticas Descritivas (desde 2000)")
print("=" * 60)
display(df_daily.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(4))

In [ ]:
# Carregar dados SGS - Monthly
df_monthly = sgs.read('ibc_br_bruto', 'ibc_br_dessaz', 'igp_m')

print("SGS MONTHLY - Estatisticas Descritivas")
print("=" * 60)
display(df_monthly.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(4))

## 3. Analise de Volatilidade

In [ ]:
# Volatilidade do Dolar (desvio padrao rolling 21 dias)
dolar = sgs.read('dolar_ptax', start='2000')['value']
dolar_ret = dolar.pct_change() * 100
dolar_vol = dolar_ret.rolling(window=21).std()

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(dolar.index, dolar, linewidth=0.8)
axes[0].set_title('Dolar PTAX - Cotacao (desde 2000)')
axes[0].set_ylabel('R$')

axes[1].fill_between(dolar_vol.index, 0, dolar_vol, alpha=0.5)
axes[1].set_title('Dolar PTAX - Volatilidade (Desvio Padrao Rolling 21 dias)')
axes[1].set_ylabel('Volatilidade (%)')
axes[1].set_xlabel('Data')

plt.tight_layout()
plt.show()

print(f"Volatilidade Media (21d): {dolar_vol.mean():.4f}%")
print(f"Volatilidade Max (21d): {dolar_vol.max():.4f}%")
print(f"Data de maior volatilidade: {dolar_vol.idxmax().strftime('%Y-%m-%d')}")

In [ ]:
# Volatilidade da Selic
selic = sgs.read('selic')['value']
selic_vol = selic.rolling(window=63).std()  # ~3 meses

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(selic.index, selic, linewidth=0.8, color='darkblue')
axes[0].set_title('Meta Selic - Taxa')
axes[0].set_ylabel('% a.a.')

axes[1].fill_between(selic_vol.index, 0, selic_vol, alpha=0.5, color='darkblue')
axes[1].set_title('Meta Selic - Volatilidade (Desvio Padrao Rolling 63 dias)')
axes[1].set_ylabel('Volatilidade (p.p.)')
axes[1].set_xlabel('Data')

plt.tight_layout()
plt.show()

### 3.1 Medias Moveis

In [ ]:
# Medias moveis - Dolar
dolar = sgs.read('dolar_ptax', start='2000')['value']

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(dolar.index, dolar, linewidth=0.5, alpha=0.7, label='Dolar PTAX')
ax.plot(dolar.index, dolar.rolling(21).mean(), linewidth=1.5, label='MM 21d (1 mes)')
ax.plot(dolar.index, dolar.rolling(63).mean(), linewidth=1.5, label='MM 63d (3 meses)')
ax.plot(dolar.index, dolar.rolling(252).mean(), linewidth=2, label='MM 252d (1 ano)')

ax.set_title('Dolar PTAX - Medias Moveis')
ax.set_ylabel('R$')
ax.set_xlabel('Data')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Medias moveis - IBC-Br
ibc = sgs.read('ibc_br_dessaz')['value']

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(ibc.index, ibc, linewidth=1, alpha=0.7, label='IBC-Br Dessaz', marker='o', markersize=2)
ax.plot(ibc.index, ibc.rolling(3).mean(), linewidth=1.5, label='MM 3 meses')
ax.plot(ibc.index, ibc.rolling(12).mean(), linewidth=2, label='MM 12 meses')

ax.set_title('IBC-Br Dessazonalizado - Medias Moveis')
ax.set_ylabel('Indice')
ax.set_xlabel('Data')
ax.legend()
plt.tight_layout()
plt.show()

### 3.2 Decomposicao Sazonal

In [ ]:
try:
    from statsmodels.tsa.seasonal import seasonal_decompose
    
    igpm = sgs.read('igp_m', start='2000')['value']
    
    decomposition = seasonal_decompose(igpm, model='additive', period=12)
    
    fig, axes = plt.subplots(4, 1, figsize=(14, 12))
    
    decomposition.observed.plot(ax=axes[0], title='IGP-M - Serie Original (desde 2000)')
    decomposition.trend.plot(ax=axes[1], title='Tendencia')
    decomposition.seasonal.plot(ax=axes[2], title='Sazonalidade')
    decomposition.resid.plot(ax=axes[3], title='Residuo')
    
    for ax in axes:
        ax.set_xlabel('')
    axes[3].set_xlabel('Data')
    
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("statsmodels nao instalado. Instale com: pip install statsmodels")

## 4. Correlacoes

In [ ]:
# Matriz de correlacao - SGS Daily
df_daily = sgs.read('selic', 'cdi', 'dolar_ptax', 'euro_ptax', start='2000')
corr_daily = df_daily.corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_daily, dtype=bool))
sns.heatmap(corr_daily, mask=mask, annot=True, cmap='RdBu_r', center=0,
            fmt='.3f', square=True, linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Matriz de Correlacao - SGS Daily')
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de correlacao - SGS Monthly
df_monthly = sgs.read('ibc_br_bruto', 'ibc_br_dessaz', 'igp_m')
corr_monthly = df_monthly.corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_monthly, dtype=bool))
sns.heatmap(corr_monthly, mask=mask, annot=True, cmap='RdBu_r', center=0,
            fmt='.3f', square=True, linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Matriz de Correlacao - SGS Monthly')
plt.tight_layout()
plt.show()

### 4.1 Correlacao Defasada (Lag Correlation)

In [ ]:
def lag_correlation(series1, series2, max_lag=30):
    """Calcula correlacao entre series com diferentes defasagens."""
    correlations = []
    lags = range(-max_lag, max_lag + 1)
    
    for lag in lags:
        if lag < 0:
            corr = series1.shift(-lag).corr(series2)
        else:
            corr = series1.corr(series2.shift(lag))
        correlations.append(corr)
    
    return pd.Series(correlations, index=lags)

# Correlacao defasada: Selic vs Dolar
df = sgs.read('selic', 'dolar_ptax', start='2000')
lag_corr = lag_correlation(df['selic'], df['dolar_ptax'], max_lag=60)

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(lag_corr.index, lag_corr.values, alpha=0.7)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.axvline(x=0, color='red', linestyle='--', linewidth=1, alpha=0.7)
ax.set_title('Correlacao Defasada: Selic vs Dolar PTAX')
ax.set_xlabel('Lag (dias) - Negativo: Selic lidera | Positivo: Dolar lidera')
ax.set_ylabel('Correlacao')
plt.tight_layout()
plt.show()

best_lag = lag_corr.abs().idxmax()
print(f"Maior correlacao (em modulo): {lag_corr[best_lag]:.4f} com lag de {best_lag} dias")

## 5. Retornos e Variacoes

In [ ]:
# Variacoes percentuais diarias
df_daily = sgs.read('selic', 'dolar_ptax', 'euro_ptax', start='2000')
df_daily_ret = df_daily.pct_change() * 100

print("Variacoes Percentuais Diarias (%)")
print("=" * 60)
display(df_daily_ret.describe().round(4))

In [ ]:
# Histograma de retornos - Dolar
dolar_ret = sgs.read('dolar_ptax', start='2000')['value'].pct_change().dropna() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(dolar_ret, bins=100, density=True, alpha=0.7, edgecolor='black')
x = np.linspace(dolar_ret.min(), dolar_ret.max(), 100)
axes[0].plot(x, stats.norm.pdf(x, dolar_ret.mean(), dolar_ret.std()), 'r-', linewidth=2, label='Normal')
axes[0].set_title('Distribuicao dos Retornos Diarios - Dolar PTAX')
axes[0].set_xlabel('Retorno (%)')
axes[0].set_ylabel('Densidade')
axes[0].legend()

# QQ-Plot
stats.probplot(dolar_ret, dist="norm", plot=axes[1])
axes[1].set_title('QQ-Plot - Dolar PTAX')

plt.tight_layout()
plt.show()

print(f"Assimetria (Skewness): {stats.skew(dolar_ret):.4f}")
print(f"Curtose: {stats.kurtosis(dolar_ret):.4f}")
print(f"Teste Jarque-Bera (normalidade): {stats.jarque_bera(dolar_ret)}")

In [ ]:
# Retorno anual do Dolar
dolar = sgs.read('dolar_ptax', start='2000')['value']
retorno_anual = dolar.resample('YE').last().pct_change() * 100
retorno_anual = retorno_anual.dropna()
retorno_anual.index = retorno_anual.index.year

fig, ax = plt.subplots(figsize=(14, 5))
colors = ['green' if x > 0 else 'red' for x in retorno_anual.values]
ax.bar(retorno_anual.index, retorno_anual.values, color=colors, alpha=0.7, edgecolor='black')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.set_title('Retorno Anual do Dolar PTAX')
ax.set_xlabel('Ano')
ax.set_ylabel('Retorno (%)')

for ano, ret in retorno_anual.items():
    ax.text(ano, ret + (2 if ret > 0 else -4), f'{ret:.1f}%', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

### 5.1 Drawdown Analysis

In [ ]:
def calculate_drawdown(series):
    """Calcula drawdown de uma serie de precos."""
    rolling_max = series.expanding().max()
    drawdown = (series - rolling_max) / rolling_max * 100
    return drawdown

# Drawdown do Dolar
dolar = sgs.read('dolar_ptax', start='2000')['value']
dolar_dd = calculate_drawdown(dolar)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(dolar.index, dolar, linewidth=0.8)
axes[0].set_title('Dolar PTAX - Cotacao (desde 2000)')
axes[0].set_ylabel('R$')

axes[1].fill_between(dolar_dd.index, 0, dolar_dd, color='red', alpha=0.5)
axes[1].set_title('Drawdown (Queda em relacao ao pico historico)')
axes[1].set_ylabel('Drawdown (%)')
axes[1].set_xlabel('Data')

plt.tight_layout()
plt.show()

print(f"Maior Drawdown: {dolar_dd.min():.2f}%")
print(f"Data do maior drawdown: {dolar_dd.idxmin().strftime('%Y-%m-%d')}")

In [ ]:
# Drawdown do IBC-Br (atividade economica)
ibc = sgs.read('ibc_br_dessaz')['value']
ibc_dd = calculate_drawdown(ibc)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(ibc.index, ibc, linewidth=1, marker='o', markersize=2)
axes[0].set_title('IBC-Br Dessazonalizado - Indice')
axes[0].set_ylabel('Indice')

axes[1].fill_between(ibc_dd.index, 0, ibc_dd, color='red', alpha=0.5)
axes[1].set_title('Drawdown - Queda da Atividade Economica')
axes[1].set_ylabel('Drawdown (%)')
axes[1].set_xlabel('Data')

plt.tight_layout()
plt.show()

print("Top 5 maiores quedas da atividade economica:")
top_dd = ibc_dd.nsmallest(5)
for date, dd in top_dd.items():
    print(f"  {date.strftime('%Y-%m')}: {dd:.2f}%")

## 6. Expectativas de Mercado

In [ ]:
# Evolucao das expectativas de IPCA
df_ipca = expectations.read('ipca_anual')

if 'DataReferencia' in df_ipca.columns and 'Mediana' in df_ipca.columns:
    # Pegar anos recentes
    anos_ref = ['2023', '2024', '2025', '2026']
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    for ano in anos_ref:
        subset = df_ipca[df_ipca['DataReferencia'] == ano]
        if len(subset) > 0:
            mediana_serie = subset.groupby(subset.index)['Mediana'].mean()
            ax.plot(mediana_serie.index, mediana_serie, label=f'IPCA {ano}', linewidth=1)
    
    ax.set_title('Evolucao das Expectativas de IPCA Anual')
    ax.set_xlabel('Data da Pesquisa')
    ax.set_ylabel('Expectativa IPCA (%)')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Dispersao das expectativas (incerteza)
if 'DesvioPadrao' in df_ipca.columns:
    dispersao = df_ipca.groupby(df_ipca.index)['DesvioPadrao'].mean()
    
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.fill_between(dispersao.index, 0, dispersao, alpha=0.5)
    ax.plot(dispersao.index, dispersao, linewidth=0.5)
    ax.set_title('Dispersao das Expectativas de IPCA (Desvio Padrao)')
    ax.set_xlabel('Data')
    ax.set_ylabel('Desvio Padrao')
    plt.tight_layout()
    plt.show()
    
    print("Periodos de maior incerteza (top 10):")
    top_dispersao = dispersao.nlargest(10)
    for date, val in top_dispersao.items():
        print(f"  {date.strftime('%Y-%m-%d')}: {val:.4f}")

## 7. Mercado de Trabalho

### 7.1 IPEA - Series Agregadas

In [ ]:
# Carregar dados IPEA
df_ipea = ipea.read('caged_saldo', 'caged_admissoes', 'caged_desligamentos', 'taxa_desemprego')

print("IPEA - Estatisticas Descritivas")
print("=" * 60)
display(df_ipea.describe().round(2))

In [ ]:
# Saldo CAGED vs Taxa de Desemprego
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Serie temporal
ax1 = axes[0]
ax2 = ax1.twinx()

ax1.plot(df_ipea.index, df_ipea['caged_saldo'], 'b-', linewidth=1, label='Saldo CAGED')
ax2.plot(df_ipea.index, df_ipea['taxa_desemprego'], 'r-', linewidth=1, label='Taxa Desemprego')

ax1.set_xlabel('Data')
ax1.set_ylabel('Saldo CAGED', color='b')
ax2.set_ylabel('Taxa Desemprego (%)', color='r')
ax1.axhline(y=0, color='blue', linestyle='--', alpha=0.3)
axes[0].set_title('Saldo CAGED vs Taxa de Desemprego')

# Scatter
axes[1].scatter(df_ipea['caged_saldo'], df_ipea['taxa_desemprego'], alpha=0.5)
axes[1].set_xlabel('Saldo CAGED')
axes[1].set_ylabel('Taxa Desemprego (%)')
corr = df_ipea['caged_saldo'].corr(df_ipea['taxa_desemprego'])
axes[1].set_title(f'Correlacao: {corr:.3f}')

plt.tight_layout()
plt.show()

In [ ]:
# Tendencia do saldo CAGED
saldo = ipea.read('caged_saldo')['value']

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(saldo.index, saldo, linewidth=0.5, alpha=0.7, label='Saldo Mensal')
ax.plot(saldo.index, saldo.rolling(3).mean(), linewidth=1.5, label='MM 3 meses')
ax.plot(saldo.index, saldo.rolling(12).mean(), linewidth=2, label='MM 12 meses')

ax.axhline(y=0, color='red', linestyle='--', linewidth=0.8, alpha=0.7)
ax.set_title('Saldo CAGED - Tendencia (Medias Moveis)')
ax.set_xlabel('Data')
ax.set_ylabel('Saldo')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1000:.0f}k'))
plt.tight_layout()
plt.show()

In [ ]:
# Sazonalidade - Saldo medio por mes do ano
saldo = ipea.read('caged_saldo')['value']
saldo_mensal = saldo.groupby(saldo.index.month).mean()

fig, ax = plt.subplots(figsize=(12, 5))

colors = ['green' if x > 0 else 'red' for x in saldo_mensal]
ax.bar(range(1, 13), saldo_mensal, color=colors, edgecolor='black')

ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun',
                    'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez'])
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.set_title('Saldo CAGED - Sazonalidade (Media por Mes)')
ax.set_xlabel('Mes')
ax.set_ylabel('Saldo Medio')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1000:.0f}k'))
plt.tight_layout()
plt.show()

### 7.2 CAGED - Microdados

In [ ]:
# Saldo mensal 2025 (usando metodo de agregacao)
try:
    df_saldo = caged.saldo_mensal(year=2025)
    
    fig, ax = plt.subplots(figsize=(12, 5))
    colors = ['green' if x > 0 else 'red' for x in df_saldo['saldo']]
    ax.bar(df_saldo['mes'], df_saldo['saldo'], color=colors, edgecolor='black')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax.set_title('Saldo CAGED por Mes - 2025 (Microdados)')
    ax.set_xlabel('Mes')
    ax.set_ylabel('Saldo')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1000:.0f}k'))
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Dados CAGED nao disponiveis: {e}")

In [ ]:
# Saldo por setor economico
try:
    df_setor = caged.saldo_por_setor(year=2025)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Top 5 positivos
    top_pos = df_setor.nlargest(5, 'saldo')
    axes[0].barh(top_pos['setor'].astype(str), top_pos['saldo'], color='green')
    axes[0].set_title('Top 5 Setores - Maior Saldo Positivo')
    axes[0].set_xlabel('Saldo')
    axes[0].invert_yaxis()
    
    # Top 5 negativos
    top_neg = df_setor.nsmallest(5, 'saldo')
    axes[1].barh(top_neg['setor'].astype(str), top_neg['saldo'], color='red')
    axes[1].set_title('Top 5 Setores - Maior Saldo Negativo')
    axes[1].set_xlabel('Saldo')
    axes[1].invert_yaxis()
    
    plt.tight_layout()
    plt.show()
    
    print("Saldo por Setor (Top 10):")
    display(df_setor.head(10))
except Exception as e:
    print(f"Erro: {e}")

In [ ]:
# Analise de um mes especifico
try:
    df_caged = caged.read(year=2025, month=10)
    
    # Distribuicao salarial
    salarios = df_caged['salário'].dropna()
    salarios = salarios[(salarios > 0) & (salarios < salarios.quantile(0.99))]
    
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.hist(salarios, bins=50, edgecolor='black', alpha=0.7)
    ax.axvline(x=salarios.median(), color='red', linestyle='--', label=f'Mediana: R$ {salarios.median():,.0f}')
    ax.axvline(x=salarios.mean(), color='blue', linestyle='--', label=f'Media: R$ {salarios.mean():,.0f}')
    ax.set_title('Distribuicao Salarial - CAGED 2025-10')
    ax.set_xlabel('Salario (R$)')
    ax.set_ylabel('Frequencia')
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    print(f"Salario mediano: R$ {salarios.median():,.2f}")
    print(f"Salario medio: R$ {salarios.mean():,.2f}")
except Exception as e:
    print(f"Erro: {e}")

In [ ]:
# Heatmap: Saldo por UF e Mes (usando SQL direto)
try:
    df_pivot = qe.sql("""
        SELECT 
            uf,
            CAST(competênciamov % 100 AS INTEGER) as mes,  -- Extração correta do mês
            SUM(saldomovimentação) as saldo
        FROM '{raw}/mte/caged/cagedmov_2025-*.parquet'
        GROUP BY 1, 2
        ORDER BY 1, 2
    """)
    
    pivot = df_pivot.pivot(index='uf', columns='mes', values='saldo')
    
    # Top 10 UFs por movimentacao
    top_ufs = pivot.sum(axis=1).abs().nlargest(10).index
    pivot_top = pivot.loc[top_ufs]
    
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.heatmap(pivot_top, cmap='RdYlGn', center=0, annot=True, fmt='.0f',
                linewidths=0.5, ax=ax, cbar_kws={'label': 'Saldo'})
    ax.set_title('Saldo CAGED por UF e Mes - 2025 (Top 10 UFs)')
    ax.set_xlabel('Mes')
    ax.set_ylabel('UF')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Erro: {e}")